# TinyMoreh — GPU Training

This notebook trains the TinyMoreh transformer on a free Colab GPU.
It clones the repo, downloads the data, and trains with larger hyperparameters
than the local CPU run.

**Before running:** Go to Runtime → Change runtime type → select **T4 GPU**.

## 1. Setup — Clone repo and download data

In [ ]:
# Clone your repo
!git clone https://github.com/Robespierre17/tiny-moreh.git
%cd tiny-moreh

In [ ]:
# Download Maimonides corpus
!python data/download_data.py

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU")

## 2. Train with larger hyperparameters

Compared to the local CPU run:
- `d_model`: 128 → 256
- `num_heads`: 4 → 8
- `num_layers`: 4 → 6
- `block_size`: 128 → 256
- `batch_size`: 32 → 64
- `max_steps`: 5000 → 10000

In [ ]:
import sys
import os
sys.path.insert(0, 'src')

import time
import torch
from data import load_corpus, prepare_data, get_batch
from model import TinyMoreh

# --- Larger hyperparameters for GPU training ---
CONFIG = {
    "d_model": 256,
    "num_heads": 8,
    "num_layers": 6,
    "block_size": 256,
    "dropout": 0.1,
    "batch_size": 64,
    "learning_rate": 3e-4,
    "max_steps": 10000,
    "eval_interval": 500,
    "eval_steps": 20,
    "generate_every": 2000,
    "generate_tokens": 300,
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

# Load data
text = load_corpus(data_dir="data")
tokenizer, train_data, val_data = prepare_data(text)
train_data = train_data.to(device)
val_data = val_data.to(device)

print(f"Corpus: {len(text):,} characters")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens: {len(val_data):,}")

# Create model
model = TinyMoreh(
    vocab_size=tokenizer.vocab_size,
    d_model=CONFIG["d_model"],
    num_heads=CONFIG["num_heads"],
    num_layers=CONFIG["num_layers"],
    block_size=CONFIG["block_size"],
    dropout=CONFIG["dropout"],
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

In [ ]:
def estimate_loss(model, train_data, val_data, config, device):
    model.eval()
    losses = {}
    for name, data in [("train", train_data), ("val", val_data)]:
        total = 0.0
        for _ in range(config["eval_steps"]):
            x, y = get_batch(data, config["block_size"], config["batch_size"], device)
            _, loss = model(x, y)
            total += loss.item()
        losses[name] = total / config["eval_steps"]
    model.train()
    return losses


def generate_sample(model, tokenizer, device, num_tokens=300):
    model.eval()
    start = torch.zeros((1, 1), dtype=torch.long, device=device)
    tokens = model.generate(start, max_new_tokens=num_tokens, temperature=0.8)
    text = tokenizer.decode(tokens[0].tolist())
    model.train()
    return text


# --- Training loop ---
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"])

print(f"Training for {CONFIG['max_steps']} steps on {device}...")
print("-" * 60)

start_time = time.time()

for step in range(CONFIG["max_steps"]):
    if step % CONFIG["eval_interval"] == 0 or step == CONFIG["max_steps"] - 1:
        losses = estimate_loss(model, train_data, val_data, CONFIG, device)
        elapsed = time.time() - start_time
        print(f"Step {step:>5d} | train: {losses['train']:.4f} | val: {losses['val']:.4f} | {elapsed:.0f}s")

    if step > 0 and step % CONFIG["generate_every"] == 0:
        sample = generate_sample(model, tokenizer, device)
        print(f"\n--- Sample at step {step} ---")
        print(sample)
        print("--- End sample ---\n")

    x, y = get_batch(train_data, CONFIG["block_size"], CONFIG["batch_size"], device)
    logits, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

elapsed = time.time() - start_time
print(f"\nTraining complete in {elapsed:.0f}s")

## 3. Generate samples

In [ ]:
prompts = [
    "The nature of God",
    "The prophet spoke",
    "It is known that",
    "The law commands",
]

model.eval()
for prompt in prompts:
    tokens = [tokenizer.stoi[ch] for ch in prompt if ch in tokenizer.stoi]
    idx = torch.tensor([tokens], dtype=torch.long, device=device)
    output = model.generate(idx, max_new_tokens=300, temperature=0.8)
    text = tokenizer.decode(output[0].tolist())
    print(f"\nPrompt: \"{prompt}\"")
    print("-" * 40)
    print(text)
    print()

## 4. Save checkpoint and download

In [ ]:
# Save checkpoint
os.makedirs("checkpoints", exist_ok=True)
save_path = "checkpoints/tiny_moreh_gpu.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "config": CONFIG,
    "vocab_size": tokenizer.vocab_size,
    "stoi": tokenizer.stoi,
    "itos": tokenizer.itos,
}, save_path)

print(f"Saved to {save_path}")
print(f"File size: {os.path.getsize(save_path) / 1024 / 1024:.1f} MB")

In [ ]:
# Download to your computer
from google.colab import files
files.download(save_path)

## Done!

After downloading `tiny_moreh_gpu.pt`:
1. Move it to `tiny-moreh/checkpoints/` on your Mac
2. Run: `python src/generate.py --checkpoint checkpoints/tiny_moreh_gpu.pt --prompt "The nature of God"`